# Backend API Test Notebook
Use this notebook to smoke-test FastAPI endpoints.

Set `API_BASE` in the next cell to either your Render URL or `http://localhost:3000` for local dev.

> **Note:** The Render free tier spins down after 15 minutes of inactivity. The first request after a cold start can take 30–60 seconds — just wait and retry.


In [1]:
import json
from typing import Any
import requests

LOCAL_URL = "http://localhost:3000"
RENDER_URL = "https://london-explorer.onrender.com"  # ← paste your Render URL here
TIMEOUT_SECONDS = 30

# Auto-select: use local if it's up, otherwise fall back to Render
try:
    requests.get(f"{LOCAL_URL}/health", timeout=2)
    API_BASE = LOCAL_URL
    print(f"✓ Local server detected — using {API_BASE}")
except requests.exceptions.ConnectionError:
    API_BASE = RENDER_URL
    print(f"✓ Local server not running — using {API_BASE}")


✓ Local server detected — using http://localhost:3000


In [3]:


def call_api(path: str, params: dict[str, Any] | None = None) -> Any:
    url = f"{API_BASE}{path}"
    response = requests.get(url, params=params, timeout=TIMEOUT_SECONDS)
    try:
        response.raise_for_status()
    except requests.HTTPError as exc:
        detail = response.text
        raise requests.HTTPError(f"{exc}\nResponse body: {detail}") from exc
    return response.json()

def preview(payload: Any, max_items: int = 3):
    if isinstance(payload, dict) and "data" in payload and isinstance(payload["data"], list):
        data = payload["data"]
        summary = {k: v for k, v in payload.items() if k != "data"}
        print("Summary:")
        print(json.dumps(summary, indent=2))
        print("\nData preview:")
        print(json.dumps(data[:max_items], indent=2))
        print(f"\nData length: {len(data)}")
        return

    print(json.dumps(payload, indent=2))


## 1) Health Check

In [4]:
health = call_api("/health")
preview(health)

{
  "status": "ok"
}


## 3) Nearby Endpoint

In [13]:
nearby_params = {
    "lat": 51.5074,
    "lng": -0.1278,
    "radius_m": 1000,
    "cuisine": "",
    "cost": "",
    "venue_type": "",
    "score_basis": 0,
    "rank_threshold": 0,
    "page": 1,
}

nearby = call_api("/api/nearby", params=nearby_params)
preview(nearby)

Summary:
{
  "page": 1,
  "page_size": 80
}

Data preview:
[
  {
    "id": "ChIJbULaEJIFdkgRZW6RTml3Npw",
    "display_name": "Zylia",
    "lat": 51.51023610000001,
    "lon": -0.1242071,
    "cuisine_type": "Unspecified",
    "venue_type": "Dine-In",
    "cost": "40+",
    "rating": 5.0,
    "user_rating_count": 172,
    "operational": true,
    "rank": 0.9947295784950256
  },
  {
    "id": "ChIJnd4G6-YFdkgRlb589_lLHM0",
    "display_name": "Vasiniko\ud83c\udf55",
    "lat": 51.5114849,
    "lon": -0.1209963999999999,
    "cuisine_type": "Pizza",
    "venue_type": "Dine-In",
    "cost": "20+",
    "rating": 4.900000095367432,
    "user_rating_count": 6806,
    "operational": true,
    "rank": 0.9928964376449585
  },
  {
    "id": "ChIJp9exgRcFdkgRpjsnrbLsyto",
    "display_name": "Brother Marcus Covent Garden",
    "lat": 51.5127993,
    "lon": -0.1263622,
    "cuisine_type": "Mediterranean",
    "venue_type": "Dine-In",
    "cost": "20+",
    "rating": 4.900000095367432,
    "user_ra

## 4) Place Detail Endpoint
Run the next cell after running nearby/tiles so you can pick a real place id.

In [14]:
sample_place_id = nearby.get("data", [{}])[0].get("id") if isinstance(nearby, dict) else None
sample_place_id

'ChIJbULaEJIFdkgRZW6RTml3Npw'

In [15]:
if not sample_place_id:
    raise ValueError("No place id available. Set sample_place_id manually and retry.")

place = call_api(f"/api/place/{sample_place_id}")
preview(place)

{
  "id": "ChIJbULaEJIFdkgRZW6RTml3Npw",
  "display_name": "Zylia",
  "primary_type_display_name": "Restaurant",
  "rating": 5.0,
  "user_rating_count": 172,
  "short_formatted_address": "6 Bedford St, London",
  "google_maps_uri": "https://maps.google.com/?cid=11256315612832558693&g_mp=Cilnb29nbGUubWFwcy5wbGFjZXMudjEuUGxhY2VzLlNlYXJjaE5lYXJieRACGAQgAA",
  "website_uri": "https://zyliataverna.com/",
  "types": "['restaurant', 'food', 'point_of_interest', 'establishment']",
  "primary_type": "restaurant",
  "is_chain": false,
  "predicted_type": null,
  "cuisine_type": "Unspecified",
  "venue_type": "Dine-In",
  "lat": 51.51023610000001,
  "lon": -0.1242071,
  "h3_r10": "8a194ad324c7fff",
  "pcd": "WC2E 9HZ",
  "areacode": "WC2E",
  "wheelchair_access": null,
  "operational": true,
  "cost": "40+",
  "wilson_1": 0.9781530499458313,
  "normal_1": 0.9947295784950256,
  "tier": 4,
  "tier_d": 4,
  "tier_independent": 4
}


## 5) Boundary and Street Spatial Search

This chapter tests the geometry-aware variants of the top places and restaurant list endpoints.

- Boundary searches use `ST_Covers` and require Polygon or MultiPolygon GeoJSON.
- Street searches use `ST_DWithin` and require LineString or MultiLineString GeoJSON plus `radius_m`.
- The geometry payload is sent as a JSON-encoded query parameter.

In [5]:
def geometry_params(search_type: str, geometry: dict[str, Any], radius_m: int | None = None) -> dict[str, Any]:
    params: dict[str, Any] = {
        "search_type": search_type,
        "geometry": json.dumps(geometry, separators=(",", ":")),
        "cuisine": "",
        "cost": "",
        "venue_type": "",
        "score_basis": 0,
        "score_tier": 0,
    }
    if radius_m is not None:
        params["radius_m"] = radius_m
    return params

boundary_geometry = {
    "type": "Polygon",
    "coordinates": [[
        [-0.15, 51.50],
        [-0.10, 51.50],
        [-0.10, 51.52],
        [-0.15, 51.52],
        [-0.15, 51.50],
    ]],
}

street_geometry = {
    "type": "LineString",
    "coordinates": [
        [-0.142, 51.501],
        [-0.130, 51.505],
        [-0.118, 51.510],
    ],
}


def call_api_expect_status(path: str, params: dict[str, Any], expected_status: int) -> requests.Response:
    response = requests.get(f"{API_BASE}{path}", params=params, timeout=TIMEOUT_SECONDS)
    if response.status_code != expected_status:
        raise AssertionError(
            f"Expected HTTP {expected_status}, got {response.status_code}: {response.text}"
        )
    return response


boundary_params = geometry_params("boundary", boundary_geometry)
boundary_top_places = call_api("/api/places/top", params={**boundary_params, "limit": 10})
boundary_restaurants = call_api(
    "/api/places/list",
    params={**boundary_params, "page": 1, "page_size": 10},
)

print("Boundary top places:")
preview(boundary_top_places)
print("\nBoundary restaurant list:")
preview(boundary_restaurants)

street_params = geometry_params("street", street_geometry, radius_m=250)
street_top_places = call_api("/api/places/top", params={**street_params, "limit": 10})
street_restaurants = call_api(
    "/api/places/list",
    params={**street_params, "page": 1, "page_size": 10},
)

print("Street top places:")
preview(street_top_places)
print("\nStreet restaurant list:")
preview(street_restaurants)

HTTPError: 500 Server Error: Internal Server Error for url: http://localhost:3000/api/places/top?search_type=boundary&geometry=%7B%22type%22%3A%22Polygon%22%2C%22coordinates%22%3A%5B%5B%5B-0.15%2C51.5%5D%2C%5B-0.1%2C51.5%5D%2C%5B-0.1%2C51.52%5D%2C%5B-0.15%2C51.52%5D%2C%5B-0.15%2C51.5%5D%5D%5D%7D&cuisine=&cost=&venue_type=&score_basis=0&score_tier=0&limit=10
Response body: Internal Server Error

In [ ]:
invalid_geometry_params = geometry_params(
    "boundary",
    {"type": "LineString", "coordinates": [[-0.13, 51.50], [-0.12, 51.51]]},
)
call_api_expect_status("/api/places/top", invalid_geometry_params, 422)

missing_radius_params = geometry_params("street", street_geometry)
call_api_expect_status("/api/places/list", missing_radius_params, 422)

print("Validation checks passed: unsupported geometry and missing street radius return HTTP 422.")